# שלב 02 — בניית גרף הרשת (Trip-Adjacency Graph)

כאן אנחנו הופכים את לוח הזמנים לגרף. המודל שבחרנו הוא **גרף נסיעות**:

- **צומת** = תחנה פעילה (תחנה שמופיעה בלפחות נסיעה אחת).
- **קשת מכוונת** `u → v` = קיימת נסיעה שבה `v` היא התחנה הבאה מיד אחרי `u`.
- **משקל הקשת** = מספר הנסיעות שמשתמשות במקטע `u → v` (תדירות).

הקובץ `stop_times.txt` ענק (כ‑15.7 מיליון שורות), ולכן אנחנו סורקים אותו שורה‑שורה במקום לטעון אותו לזיכרון.

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas networkx

## הגדרת נתיבים

קוראים את התחנות המנוקות משלב 01, ואת `stop_times.txt` הגולמי.

In [ ]:
from pathlib import Path
import csv, json, pickle
from collections import defaultdict
import pandas as pd
import networkx as nx


def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
BASE = ROOT / "public_transport_network_notebooks"
STOPS_CLEAN = BASE / "outputs" / "01_data_preparation" / "stops_clean.csv"
STOP_TIMES = ROOT / "israel-public-transportation" / "stop_times.txt"
OUT_DIR = BASE / "outputs" / "02_graph_construction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv.field_size_limit(10_000_000)  # שורות ארוכות מאוד ב-stop_times
print("OUT_DIR:", OUT_DIR)

## תכונות לכל תחנה

טוענים את התכונות של כל תחנה (שם, קואורדינטות, אזור, מטרופולין) כדי לצרף אותן לצמתים בגרף.

In [ ]:
def load_stop_attributes():
    stops = pd.read_csv(STOPS_CLEAN, dtype=str, encoding="utf-8-sig")
    attr = {}
    for _, r in stops.iterrows():
        sid = r["stop_id"]
        try:
            lat = float(r["stop_lat"]) if r.get("stop_lat") not in (None, "", "nan") else None
            lon = float(r["stop_lon"]) if r.get("stop_lon") not in (None, "", "nan") else None
        except (TypeError, ValueError):
            lat = lon = None
        attr[sid] = {
            "stop_name": r.get("stop_name", "") or "",
            "lat": lat,
            "lon": lon,
            "region": r.get("region", "") or "",
            "metro": r.get("metro", "") or "",
        }
    return attr


attr = load_stop_attributes()
print(f"{len(attr):,} תחנות עם תכונות")

## סריקת הנסיעות ובניית מקטעים

סורקים את `stop_times.txt` שורה‑שורה. הקובץ ממוין לפי `trip_id` ואז `stop_sequence` (תקן GTFS), כך ששתי שורות עוקבות מאותה נסיעה מגדירות מקטע `u → v`. סופרים כמה פעמים כל מקטע מופיע — זה המשקל.

In [ ]:
def stream_trip_edges(stop_times_path):
    edge_count = defaultdict(int)
    active_stops = set()
    trips_seen = set()
    rows_read = 0
    stops_per_trip = defaultdict(int)

    with open(stop_times_path, encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index("trip_id")
        si = header.index("stop_id")

        prev_trip = None
        prev_stop = None
        for row in reader:
            rows_read += 1
            trip = row[ti]
            stop = row[si]
            active_stops.add(stop)
            trips_seen.add(trip)
            stops_per_trip[trip] += 1

            if trip == prev_trip and prev_stop is not None and prev_stop != stop:
                edge_count[(prev_stop, stop)] += 1

            prev_trip = trip
            prev_stop = stop

            if rows_read % 2_000_000 == 0:
                print(f"    עיבד {rows_read:,} שורות, {len(edge_count):,} מקטעים ייחודיים ...")

    spt = list(stops_per_trip.values())
    build_stats = {
        "stop_times_rows": rows_read,
        "active_stops": len(active_stops),
        "active_trips": len(trips_seen),
        "directed_edges": len(edge_count),
        "min_stops_per_trip": int(min(spt)) if spt else 0,
        "mean_stops_per_trip": round(sum(spt) / len(spt), 2) if spt else 0,
        "max_stops_per_trip": int(max(spt)) if spt else 0,
    }
    return edge_count, active_stops, build_stats


print("סורק את stop_times.txt (קובץ גדול, נא להמתין) ...")
edge_count, active_stops, build_stats = stream_trip_edges(STOP_TIMES)
build_stats

## הרכבת הגרפים

מהמקטעים בונים שני גרפים: **מכוון** (`DiGraph`, שומר את כיוון הנסיעה) ו**לא מכוון** (`Graph`, מאחד שני הכיוונים). הגרף הלא מכוון נחוץ לחישובי Bridges, Articulation Points וקהילות בהמשך.

In [ ]:
def build_graphs(edge_count, attr):
    default_attr = {"stop_name": "", "lat": None, "lon": None, "region": "", "metro": ""}

    D = nx.DiGraph()
    for (u, v), c in edge_count.items():
        D.add_edge(u, v, weight=c)
    for n in D.nodes():
        D.nodes[n].update(attr.get(n, default_attr))

    G = nx.Graph()
    for u, v, data in D.edges(data=True):
        w = data["weight"]
        if G.has_edge(u, v):
            G[u][v]["weight"] += w
        else:
            G.add_edge(u, v, weight=w)
    for n in G.nodes():
        G.nodes[n].update(attr.get(n, default_attr))

    return G, D


G, D = build_graphs(edge_count, attr)
print(f"גרף לא מכוון: {G.number_of_nodes():,} צמתים, {G.number_of_edges():,} קשתות")
print(f"גרף מכוון:    {D.number_of_nodes():,} צמתים, {D.number_of_edges():,} קשתות")

## שמירת הגרפים והפלטים

שומרים את הגרפים כ‑pickle (לטעינה מהירה בשלבים הבאים), וכן `nodes.csv`, `edges.csv` וסיכום בנייה.

In [ ]:
with open(OUT_DIR / "graph_undirected.pkl", "wb") as f:
    pickle.dump(G, f)
with open(OUT_DIR / "graph_directed.pkl", "wb") as f:
    pickle.dump(D, f)

node_rows = [{"stop_id": nid, **data} for nid, data in G.nodes(data=True)]
pd.DataFrame(node_rows).to_csv(OUT_DIR / "nodes.csv", index=False, encoding="utf-8-sig")

edge_rows = [{"from_stop": u, "to_stop": v, "trip_frequency": data["weight"]}
             for u, v, data in D.edges(data=True)]
pd.DataFrame(edge_rows).to_csv(OUT_DIR / "edges.csv", index=False, encoding="utf-8-sig")

avg_degree = round(sum(d for _, d in G.degree()) / G.number_of_nodes(), 2)
summary = {
    "graph_type": "trip_adjacency",
    "num_nodes": G.number_of_nodes(),
    "num_edges_undirected": G.number_of_edges(),
    "num_edges_directed": D.number_of_edges(),
    "avg_degree": avg_degree,
    "density": round(nx.density(G), 6),
    "build_stats": build_stats,
}
with open(OUT_DIR / "graph_build_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("דרגה ממוצעת:", avg_degree, "| צפיפות:", summary["density"])
print("נשמר ב:", OUT_DIR)